# S2S2Fun Tutorial: Training with Your Own Data

This notebook walks through two training modes:

- **S2S2Fun** — standard training when all your proteins come from one family
- **S2S2Fun-DA** — domain-adversarial training when proteins come from two structurally different families that share the same function (e.g., MRP and CRBP both bind retinal)

### Prerequisites

```bash
conda env create -f environment.yaml
conda activate absdesign
```

### What you need before starting

1. AlphaFold3 output directory (see Step 1)
2. A CSV file with your protein IDs and measured function values (see Step 2)

---
## Step 1 — AlphaFold3 Output Format

Run AF3 on each protein sequence **with its ligand** (protein–ligand complex input). 
S2S2Fun reads `pair_embeddings` and `single_embeddings` from the `.npz` files that AF3 saves when `save_embeddings=true`.

Your AF3 output directory must follow this layout:

```
af3_output/
├── proteinA/
│   ├── proteinA_ranking_scores.csv     # AF3 ranking scores per seed
│   ├── proteinA_model.pdb              # best predicted structure
│   └── seed-1_embeddings/
│       └── proteinA_embeddings.npz     # pair + single embeddings
├── proteinB/
│   └── ...
```

The `_ranking_scores.csv` must contain columns `ranking_score` and `seed`.  
S2S2Fun automatically picks the seed with the highest `ranking_score`.

> **Ligand token count** — you must know `Nligand` for your system:
> | System | Nligand |
> |---|---|
> | Retinal-binding proteins (RET) | 20 |
> | GFP-like chromophore | 3 |
> | HBI brightness | 18 |
> | HBI stability (protein-only) | 0 |
> | Your custom ligand | count atoms in SMILES / CCD |

> **PDB file** — if AF3 only produces `.cif`, `inference_s2s2fun.py` will auto-convert it to `.pdb`.

---
## Step 2 — Prepare Your Data CSV

### 2a. S2S2Fun (no DANN) — single protein family

Your CSV needs these columns:

| column | description |
|---|---|
| `seq_id` | protein identifier, must match the folder name in your AF3 output dir (case-insensitive) |
| `function_value` | measured functional value (e.g., absorption peak in nm, brightness, Tm) |
| `set` | `train` or `test` |

Example:

In [ ]:
import pandas as pd
import numpy as np

# Minimal example — replace with your real data
data = {
    'seq_id':         ['protA', 'protB', 'protC', 'protD', 'protE', 'protF'],
    'function_value': [520.0,   545.0,   490.0,   570.0,   510.0,   555.0 ],
    'set':            ['train', 'train', 'train', 'train', 'test',  'test' ],
}
df = pd.DataFrame(data)
df.to_csv('my_data.csv', index=False)
df

### 2b. S2S2Fun-DA — two protein families sharing the same function

Add two extra columns:

| column | description |
|---|---|
| `crbp` | `False` = source domain (well-characterised family, used for training the task predictor) · `True` = target domain (family you want to generalise to) |
| `k_pos` | position of the key residue (e.g., Schiff-base Lys); set to any non-NaN value if known, otherwise filter these out before training |

The DANN framework trains the task predictor on **source domain only**, while the domain-confusion encoder learns representations that are invariant to the fold difference.

In [ ]:
# S2S2Fun-DA data CSV example
data_da = {
    'seq_id':         ['mrp1',  'mrp2',  'mrp3',  'mrp4',  'crbp1', 'crbp2'],
    'function_value': [520.0,   545.0,   490.0,   570.0,   440.0,   460.0 ],
    'set':            ['train', 'train', 'train', 'test',  'train', 'test' ],
    'crbp':           [False,   False,   False,   False,   True,    True  ],
    'k_pos':          [200,     200,     200,     200,     108,     108   ],
}
df_da = pd.DataFrame(data_da)
df_da.to_csv('my_data_da.csv', index=False)
df_da

---
## Step 3 — Extract AF3 Features

Before training, verify that S2S2Fun can find and load the AF3 embeddings for all your proteins.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from features import get_embeddings_file, extract_features, get_nearest_residues, cif2pdb
import numpy as np
import torch

AF3_OUT_DIR = './af3_output'   # <-- change to your AF3 output directory
LIGAND_NUM  = 20               # <-- change to match your ligand atom count
P_NUM       = 30               # nearest residues to crop (30 recommended)

emb_files = get_embeddings_file(AF3_OUT_DIR)
print(f'Found {len(emb_files)} embedding files')
for f in emb_files[:3]:
    print(' ', f)

In [ ]:
# Quick sanity-check: load one embedding and inspect shapes
sample_file = emb_files[0]
data = np.load(sample_file)
pair   = data['pair_embeddings']    # (L, L, 128)
single = data['single_embeddings']  # (L, 384)

print('pair shape  :', pair.shape)
print('single shape:', single.shape)
print('Expected total tokens (L) = len(protein_seq) + LIGAND_NUM')

---
## Step 4 — Train S2S2Fun (no domain adaptation)

Use this when **all your proteins belong to one structural family**.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
AF3_OUT_DIR      = './af3_output'   # AF3 output directory
DATA_INFO_CSV    = './my_data.csv'  # your data CSV (seq_id, function_value, set)
FUNCTION_COL     = 'function_value' # column name for measured function
LIGAND_NUM       = 20
P_NUM            = 30
INPUT_DIM        = 1280             # 512 (pair) + 768 (single)

EPOCHS           = 1000
BATCH_SIZE       = 256
LR               = 1e-4
HIDDEN_DIM       = 256
DROPOUT          = 0.1
SAVE_PATH        = './s2s2fun.pth'
TB_FOLDER        = './tb_logs'
OUT_CSV_DIR      = './csv'
TRAIN_NAME       = 'my_s2s2fun'
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from features import get_embeddings_file, cached_extract_sequences, cached_get_nearest_residues, extract_features
from models import RankNetModel
from metric import evaluate
import pandas as pd
import numpy as np
import torch
import os

def _process_file(args):
    embeddings_file, af3_out_dir, p_num, ligand_num = args
    seq_name = embeddings_file.split('/')[-3]
    data = np.load(embeddings_file, mmap_mode='r')
    pair   = data['pair_embeddings']
    single = data['single_embeddings']

    pdb_file = os.path.join(af3_out_dir, f"{seq_name.lower()}/{seq_name.lower()}_model.pdb")
    seqs = cached_extract_sequences(pdb_file)['A']
    all_tokens = len(seqs) + ligand_num

    if pair.shape[0] == all_tokens + 1:
        ligand_index = list(range(all_tokens - ligand_num, all_tokens + 1))
        ligand_index.remove(all_tokens - 5)
    else:
        ligand_index = list(range(all_tokens - ligand_num, all_tokens))

    near_index = [i - 1 for i in sorted(cached_get_nearest_residues(pdb_file, 'RET', p_num))]
    idx = np.array(near_index + ligand_index)
    feature = extract_features(
        torch.from_numpy(pair[idx][:, idx, :]),
        torch.from_numpy(single[idx, :]),
        N_protein=p_num, N_ligand=ligand_num
    )
    return seq_name, feature

def load_features(af3_out_dir, data_info, p_num=30, ligand_num=20, function_col='function_value'):
    all_seq_ids = [s.lower() for s in data_info['seq_id'].tolist()]
    emb_files = get_embeddings_file(af3_out_dir, all_seq_ids)
    assert len(emb_files) == len(data_info), \
        f"Found {len(emb_files)} embedding files but data_info has {len(data_info)} rows. " \
        f"Check that seq_ids match the AF3 output folder names."

    n = len(emb_files)
    features = torch.empty((n, 1280), dtype=torch.float32)
    labels   = [None] * n
    args_list = [(f, af3_out_dir, p_num, ligand_num) for f in emb_files]

    with ThreadPoolExecutor(max_workers=32) as ex:
        futures = {ex.submit(_process_file, a): i for i, a in enumerate(args_list)}
        for fut in as_completed(futures):
            i = futures[fut]
            seq_name, feat = fut.result()
            labels[i]   = seq_name
            features[i] = feat

    labels = [s.lower() for s in labels]
    data_info = data_info.copy()
    data_info['seq_id_lower'] = data_info['seq_id'].str.lower()
    data_info = data_info.set_index('seq_id_lower').loc[labels]

    y = np.array([data_info.loc[l, function_col] for l in labels])
    return features, y, labels, data_info

print('Functions loaded.')

In [ ]:
data_info = pd.read_csv(DATA_INFO_CSV)

all_features, y_all, all_labels, data_info_indexed = load_features(
    AF3_OUT_DIR, data_info, p_num=P_NUM, ligand_num=LIGAND_NUM, function_col=FUNCTION_COL
)

train_mask = data_info_indexed['set'] == 'train'
test_mask  = data_info_indexed['set'] == 'test'

X_train = all_features[train_mask.values]
y_train = torch.tensor(y_all[train_mask.values], dtype=torch.float32)
X_test  = all_features[test_mask.values]
y_test  = torch.tensor(y_all[test_mask.values],  dtype=torch.float32)
seqids_test = np.array(all_labels)[test_mask.values]

print(f'Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples')
print(f'Function value range: [{y_all.min():.1f}, {y_all.max():.1f}]')

In [ ]:
# Import the training loop from train_s2s2fun.py
from train_s2s2fun import train, PairDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

model = RankNetModel(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, num_blocks=3, dropout=DROPOUT)

model, history = train(
    model, X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    mode='pairs',
    val_data=(X_test, y_test, seqids_test),
    skip_equal=True,
    save_path=SAVE_PATH,
    log_every=50,
    tb_folder=TB_FOLDER,
    name=TRAIN_NAME,
    out_csv_dir=OUT_CSV_DIR,
    device=device,
)

In [ ]:
# Quick evaluation on test set
model.eval()
with torch.no_grad():
    scores = model(X_test.to(device)).cpu().numpy()

results = evaluate(y_test.numpy(), scores, k=None, skip_equal=True)
print('Test results:')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Plot training curve
import matplotlib.pyplot as plt

epochs = list(history.keys())
losses = [history[e]['loss'] for e in epochs]
accs   = [history[e]['val_pair_acc'] for e in epochs if history[e]['val_pair_acc'] is not None]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, losses)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train Loss'); ax1.set_title('RankNet Loss')
ax2.plot(epochs[:len(accs)], accs)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Pair Accuracy'); ax2.set_title('Validation Pair Accuracy')
plt.tight_layout()
plt.show()

---
## Step 5 — Train S2S2Fun-DA (Domain Adversarial)

Use this when you have **two protein families** with the same function but different folds.  
The task predictor trains only on **source domain**, while the domain-confusion encoder learns to produce fold-invariant features via the gradient reversal layer (GRL).

```
Source domain  →  Gc (encoder)  →  Gy (task predictor)  →  ranking loss
                       ↓
                    GRL (reversal)
                       ↓
Target domain  →  Gc (encoder)  →  Gd (domain classifier) → domain loss (maximised)
```

**When λ=0** (DANN disabled), S2S2Fun-DA reduces to a plain ranker trained on source+target pooled together.

In [ ]:
# ── DANN CONFIG ──────────────────────────────────────────────────────────────
AF3_OUT_DIR_DA   = './af3_output'
DATA_INFO_DA_CSV = './my_data_da.csv'
FUNCTION_COL_DA  = 'function_value'
LIGAND_NUM_DA    = 20
P_NUM_DA         = 30

NUM_EPOCHS_DA    = 2000
LAMBDA_MAX       = 1.0          # max adversarial weight; set 0.0 to disable DANN
STAGE1_EPOCH     = 500          # epochs to train on source-only before adding target
USE_DANN         = True         # False = train without domain adversarial objective
LOSS_TYPE        = 'pairwise'   # 'pointwise' | 'pairwise' | 'listwise'

TB_FOLDER_DA     = './tb_logs_da'
CHECKPOINT_DIR   = './checkpoint'
OUT_CSV_DIR_DA   = './csv_da'
TRAIN_NAME_DA    = 'my_s2s2fun_da'
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
from train_s2s2fun_da import DomainDataset, train_dann, get_feature
from models import RankNetModel, FeatureExtractor, DomainClassifier
from features import get_embeddings_file
from torch.utils.data import DataLoader

data_info_da = pd.read_csv(DATA_INFO_DA_CSV)
# filter rows without key residue annotation
data_info_da = data_info_da[data_info_da['k_pos'].notna()]

all_seq_ids = [s.lower() for s in data_info_da['seq_id'].tolist()]
emb_files_da = get_embeddings_file(AF3_OUT_DIR_DA, all_seq_ids)
assert len(emb_files_da) == len(data_info_da), \
    f"Mismatch: {len(emb_files_da)} embeddings vs {len(data_info_da)} rows in CSV"

all_labels_da, all_features_da = get_feature(
    AF3_OUT_DIR_DA, emb_files_da, p_num=P_NUM_DA, ligand_num=LIGAND_NUM_DA, diag=False, pooling=True
)
print(f'Total proteins loaded: {all_features_da.shape[0]}')

In [ ]:
# Build domain masks from CSV
y_func = np.array([
    data_info_da.loc[data_info_da['seq_id'].str.lower() == l.lower(), FUNCTION_COL_DA].values[0]
    for l in all_labels_da
])
y0 = np.mean(y_func)   # reference value for CalRankNetLoss
print(f'y0 (mean function value): {y0:.2f}')

data_info_da = data_info_da.copy()
data_info_da['seq_id_lower'] = data_info_da['seq_id'].str.lower()
all_labels_da_lower = [s.lower() for s in all_labels_da]

df_map   = data_info_da.set_index('seq_id_lower').loc[all_labels_da_lower, ['crbp', 'set']]
all_names_da = data_info_da.set_index('seq_id_lower').loc[all_labels_da_lower, 'seq_id'].tolist()

# domain_label: source=1 (MRP/non-crbp), target=0 (CRBP)
domain_label = np.array((~df_map['crbp']).astype(int).tolist())
domain_mask  = np.array(df_map['crbp'].astype(bool).tolist())   # True = target domain

# Stage 1: source train only  |  Stage 2: source train + target train
s1_mask   = np.array((~df_map['crbp']) & (df_map['set'] == 'train')).astype(bool).tolist()
s2_mask   = np.array((df_map['set'] == 'train').astype(bool).tolist())
test_mask_da = np.array((df_map['set'] == 'test')).astype(bool).tolist()

print(f"Stage-1 source train: {sum(s1_mask)}")
print(f"Stage-2 all train:    {sum(s2_mask)}")
print(f"Target (CRBP) total:  {domain_mask.sum()}")
print(f"Test set total:       {sum(test_mask_da)}")

In [ ]:
def make_domain_dataset(feature_arr, y_arr, domain_label_arr, name_arr):
    return DomainDataset(feature_arr, y_arr, domain_label_arr.astype(float), list(name_arr))

names = np.array(all_names_da)

ds_stage1  = make_domain_dataset(all_features_da[s1_mask],       y_func[s1_mask],       domain_label[s1_mask],       names[s1_mask])
ds_stage2  = make_domain_dataset(all_features_da[s2_mask],       y_func[s2_mask],       domain_label[s2_mask],       names[s2_mask])
ds_target  = make_domain_dataset(all_features_da[domain_mask],   y_func[domain_mask],   domain_label[domain_mask],   names[domain_mask])
ds_test    = make_domain_dataset(all_features_da[test_mask_da],  y_func[test_mask_da],  domain_label[test_mask_da],  names[test_mask_da])

loader_s1     = DataLoader(ds_stage1, batch_size=256,  shuffle=False)
loader_s2     = DataLoader(ds_stage2, batch_size=1028, shuffle=False)
loader_target = DataLoader(ds_target, batch_size=1028, shuffle=False)

print('Datasets ready.')

In [ ]:
INPUT_DIM_DA = 1280
device_da = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device_da)

Gf = FeatureExtractor(dim=INPUT_DIM_DA, hidden=512, dropout=0.1)
Gc = RankNetModel(input_dim=INPUT_DIM_DA)
Gd = DomainClassifier(input_dim=INPUT_DIM_DA)

train_dann(
    Gf, Gc, Gd,
    loader_s1, loader_s2, loader_target,
    num_epochs=NUM_EPOCHS_DA,
    lambda_max=LAMBDA_MAX,
    stage1_epoch=STAGE1_EPOCH,
    device=device_da,
    tb_folder=TB_FOLDER_DA,
    target_dataset=ds_target,
    test_dataset=ds_test,
    train_name=TRAIN_NAME_DA,
    ck_epoch=NUM_EPOCHS_DA - 200,
    loss_type=LOSS_TYPE,
    a=1.0,
    y0=float(y0),
    cal=False,
    sigmoid_scale=1.0,
    DANN=USE_DANN,
    checkpoint_dir=CHECKPOINT_DIR,
    out_csv_dir=OUT_CSV_DIR_DA,
)

---
## Step 6 — Inference on New Sequences

Once trained, run inference to rank new protein variants.

In [ ]:
# ── Inference with S2S2Fun (standard) ────────────────────────────────────────
from inference_s2s2fun import get_embeddings_file as inf_get_emb
from inference_s2s2fun import get_feature as inf_get_feat
from inference_s2s2fun import load_model as inf_load_model

NEW_AF3_DIR  = './af3_new_sequences'   # AF3 output for your new proteins
CHECKPOINT   = './s2s2fun.pth'
OUTPUT_CSV   = './predictions.csv'
P_NUM_INF    = 30
LIGAND_NUM_INF = 20

inf_emb_files = inf_get_emb(NEW_AF3_DIR)
inf_labels, inf_features = inf_get_feat(NEW_AF3_DIR, inf_emb_files, p_num=P_NUM_INF, ligand_num=LIGAND_NUM_INF)

device_inf = 'cuda' if torch.cuda.is_available() else 'cpu'
inf_model = inf_load_model(CHECKPOINT, device=device_inf)

with torch.no_grad():
    inf_scores = inf_model(inf_features.to(device_inf, dtype=torch.float32)).cpu().numpy()

result_df = pd.DataFrame({'seq_id': inf_labels, 'score': inf_scores})
result_df = result_df.sort_values('score', ascending=False)
result_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved predictions to {OUTPUT_CSV}')
result_df.head(10)

In [ ]:
# ── Inference with S2S2Fun-DA ─────────────────────────────────────────────────
from inference_s2s2fun_da import get_feature as da_get_feat
from inference_s2s2fun_da import load_model as da_load_model
import glob

DA_CHECKPOINT   = './checkpoint/my_s2s2fun_da_epoch_XXXX.pt'  # <-- pick best epoch
DA_AF3_DIR      = './af3_new_sequences'
DA_OUTPUT_CSV   = './predictions_da.csv'

da_emb_files = glob.glob(f'{DA_AF3_DIR}/*/*_model.cif')
da_emb_files = [glob.glob(f'{os.path.dirname(f)}/seed-1_embeddings/*.npz')[0] for f in da_emb_files]

da_labels, da_features = da_get_feat(da_emb_files, p_num=30, ligand_num=20)

device_da_inf = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Gf_inf, Gc_inf, Gd_inf = da_load_model(DA_CHECKPOINT, device_da_inf)

with torch.no_grad():
    da_feats_t = da_features.float().to(device_da_inf)
    da_emb     = Gf_inf(da_feats_t)
    da_scores  = Gc_inf(da_emb).cpu().numpy()

da_df = pd.DataFrame({'seq_id': da_labels, 'score': da_scores.squeeze()})
da_df = da_df.sort_values('score', ascending=False)
da_df.to_csv(DA_OUTPUT_CSV, index=False)
print(f'Saved DA predictions to {DA_OUTPUT_CSV}')
da_df.head(10)

---
## Step 7 — Evaluate Predictions

If you have ground-truth labels for your test set, compute all four metrics used in the paper.

In [ ]:
from metric import evaluate
import numpy as np

# Example: load your test labels and predicted scores
# y_true  = np.array([520, 545, 490, 570, 510])   # ground truth
# y_pred  = np.array([0.3, 0.8, -0.2, 1.1, 0.1]) # model scores

# results = evaluate(y_true, y_pred, k=None, skip_equal=True)
# for metric, value in results.items():
#     print(f'{metric:30s}: {value:.4f}')

---
## Command-Line Equivalents

Everything above can also be run from the terminal:

```bash
# Train S2S2Fun
python train_s2s2fun.py \
    --af3_out_dir ./af3_output \
    --train_data_info ./my_data.csv \
    --epochs 1000 \
    --lr 1e-4 \
    --save ./s2s2fun.pth \
    --tb_folder ./tb_logs \
    --out_csv_dir ./csv \
    --train_name my_run

# Train S2S2Fun-DA
python train_s2s2fun_da.py \
    --af3_out_dir ./af3_output \
    --data_info ./my_data_da.csv \
    --num_epochs 2000 \
    --loss_type pairwise \
    --DANN \
    --train_name my_da_run

# Inference
python inference_s2s2fun.py \
    --af3_out_dir ./af3_new_sequences \
    --checkpoint ./s2s2fun.pth \
    --out_file ./predictions.csv
```

Monitor training in TensorBoard:
```bash
tensorboard --logdir ./tb_logs
```